In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

df_sensors = pd.read_csv("./metadata/sdot_coords.csv")
df_sensors = df_sensors.rename(columns={"No": "id", "경도": "lon", "위도": "lat", '주소': 'address', "모델 시리얼(*)":"serial"})
df_sensors = df_sensors[["id", "serial", "lon", "lat", "address"]]

gdf_sensors = gpd.GeoDataFrame(
    df_sensors[["serial", "address"]].copy(),
    geometry=[Point(xy) for xy in zip(df_sensors["lon"], df_sensors["lat"])],
    crs="EPSG:4326"
)

gdf_sensors

,serial,address,geometry
0,V02Q1940059,서울특별시 강남구 일원동 688,POINT (127.07535 37.48953)
1,V02Q1940293,서울특별시 강남구 개포동 1221-3,POINT (127.04849 37.47640)
2,V02Q1940060,서울특별시 강남구 일원동 735-1,POINT (127.08623 37.48340)
3,V02Q1940079,서울특별시 강남구 일원동 707,POINT (127.08221 37.49410)
4,V02Q1940129,서울특별시 강남구 대치동 16-1,POINT (127.07440 37.49766)
...,...,...,...
1154,OC3CL200169,서울특별시 중랑구 신내동 256-24,POINT (127.10615 37.62027)
1155,OC3CL200225,서울특별시 중랑구 묵동 189-11,POINT (127.07281 37.60952)
1156,OC3DL2200011,서울특별시 중랑구 면목동 49-10,POINT (127.09486 37.58620)
1157,OC3DL2200010,서울특별시 중랑구 상봉동 481-2,POINT (127.09005 37.60412)


In [2]:
# add SUPABASE_STORAGE와 SUPABASE_PASSWORD from .env 
# Supabase Storage (Postgres)

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username="postgres.ugnxjiafqlqxzgypvlnl",     # ← shared pooler는 이런 형식
    password=SUPABASE_PASSWORD,
    host="aws-1-ap-southeast-2.pooler.supabase.com",
    port=6543,
    database="postgres",
    query={"sslmode": "require"},
)
engine = create_engine(url)

# Test connection
with engine.begin() as conn:
    print(conn.execute(text("select now()")).scalar())

2025-11-08 14:33:41.922624+00:00


In [3]:
gdf_sensors.to_postgis(
    name="sensor",
    con=engine,
    schema="public",
    if_exists="append", 
    index=False
)